# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrinceNacion/Starter-Notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row is one content item (page) in the anonymized search-content dataset. The data is a cross-section of 30,000 pages across 32 pseudonymized clients, and each row summarizes the page’s trailing 90-day performance ending at export time. In other words, the unit of analysis is a page-level record, not a client-day or query-day record.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path.cwd()
csv_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)

print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
print(f"Unique content_id (grain): {df['content_id'].nunique():,}")
print(f"Unique client_id count: {df['client_id'].nunique()}")
print(f"Content age range: {df['content_age_days'].min()} to {df['content_age_days'].max()} days")
print(f"Days with impressions range: {df['days_with_impressions'].min()} to {df['days_with_impressions'].max()} days")

Loaded 30,000 rows × 44 columns
Unique content_id (grain): 30,000
Unique client_id count: 32
Content age range: 90 to 564 days
Days with impressions range: 1 to 88 days


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field in the 44-column dataset belongs to exactly one bucket:

1. **Features (knowable before the decision moment, safe to use):**
   - **Content & Freshness:** `content_age_days`, `days_since_last_update`, `age_tier`, `age_tier_order`, `freshness_tier`, `word_count`, `char_count`, `word_count_tier`, `char_count_tier`.
   - **Keyword & Intent Context:** `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`.
   - **90-Day Aggregate Performance & Rates:** `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`.
   - **Prior Baseline Window (days 31–60 back):** `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`.

2. **Label / Proxies (target outcomes and direct derivative components):**
   - `trend_direction`: Categorical direction (`down`, `stable`, `up`, `new`, `flat`). The binary target `is_declining_label` is defined as `(trend_direction == "down")`.
   - `trend_pct`: Percentage change between recent and prior 30 days used to calculate `trend_direction`.
   - `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`: Recent 30-day outcome window metrics that define trend trajectory.

3. **Context (grouping, joins, and validation splits — never trained as direct features):**
   - `content_id`: Pseudonymous primary key for row identification.
   - `client_id`: Pseudonymous client identifier (32 distinct clients), used for client-holdout splits.

4. **Excluded (metadata or leakage risks — with a why for each):**
   - `provider_used`, `model_used`: Generative AI platform metadata (`openai`, `google`, `gemini-2.5-flash`, etc.) reflecting back-office generation tools rather than search signals. Excluded to prevent confounding generation vendor tooling with ranking opportunity.
   - `trend_direction`, `trend_pct`: Direct mathematical source of the target label; inclusion causes 100% target leakage.
   - `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`: Outcome window totals that define the trend direction; inclusion leaks the future outcome.

In [2]:
features = [
    "content_age_days", "days_since_last_update", "age_tier", "age_tier_order",
    "freshness_tier", "word_count", "char_count", "word_count_tier", "char_count_tier",
    "search_volume", "competition", "competition_level", "cpc", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "days_with_impressions",
    "days_with_sessions", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "impression_tier", "position_tier",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"
]

labels_and_outcome_inputs = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"
]

context_identifiers = [
    "content_id", "client_id"
]

excluded_metadata = [
    "provider_used", "model_used"
]

all_classified = set(features + labels_and_outcome_inputs + context_identifiers + excluded_metadata)
raw_columns = set(df.columns)

print(f"Total raw columns: {len(raw_columns)}")
print(f"  - Features: {len(features)}")
print(f"  - Labels & Outcome Window: {len(labels_and_outcome_inputs)}")
print(f"  - Context identifiers: {len(context_identifiers)}")
print(f"  - Excluded metadata: {len(excluded_metadata)}")
print(f"All columns accounted for without overlap: {all_classified == raw_columns}")

Total raw columns: 44
  - Features: 35
  - Labels & Outcome Window: 5
  - Context identifiers: 2
  - Excluded metadata: 2
All columns accounted for without overlap: True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.